# Step 2a · Train SAC on Pusher (Kaggle CPU)

Settings: **Accelerator → None (CPU)**, **Internet → On**.

Use **Save Version → Save & Run All (Commit)**: the run (about 5 hours for the 1M-step run) keeps going with the tab closed, and files appear in the version's **Output** tab.

The `assert` cells stop the notebook if a step silently failed (`!` shell cells never stop it on their own).

All logic lives in the repo; this notebook only clones and calls scripts.

In [ ]:
# 1. Fresh clone of the repo (always from /kaggle/working, so reruns don't nest clones)
%cd /kaggle/working
!rm -rf robot-world-model
!git clone -q https://github.com/Swayam-27/robot-world-model.git
%cd /kaggle/working/robot-world-model
!git log --oneline -1

In [ ]:
# 2. Install MuJoCo + Stable-Baselines3 (Kaggle already has PyTorch, pandas, matplotlib)
!pip install -q "gymnasium[mujoco]" stable-baselines3
!python -c "import gymnasium, mujoco, stable_baselines3, torch; print('gymnasium', gymnasium.__version__, '| mujoco', mujoco.__version__, '| sb3', stable_baselines3.__version__, '| torch', torch.__version__)"

In [ ]:
# 3. Smoke test: 3,000 steps, then watch the result (it will NOT push yet; this only proves the pipeline runs)
!python -m policies.train_sac --config configs/sac_pusher_smoke.yaml 2>&1 | tail -5
!apt-get -qq install -y libosmesa6 > /dev/null
!python -m policies.watch_checkpoints --run outputs/sac_pusher_smoke --ckpts sac_final
from pathlib import Path
for f in ["sac_final.zip", "learning_curve.png", "sac_final.seed0.gif"]:
    assert (Path("outputs/sac_pusher_smoke") / f).exists(), f"smoke test failed: {f} missing (read the output above)"
print("smoke test OK")

In [ ]:
# 4. Full run: 1,000,000 steps, checkpoint every 20,000 (about 5 hours). Log also saved to a file.
!mkdir -p outputs/sac_pusher_1m && python -m policies.train_sac --config configs/sac_pusher_1m.yaml 2>&1 | tee outputs/sac_pusher_1m/train_log.txt
assert Path("outputs/sac_pusher_1m/sac_final.zip").exists(), "training failed: read the log above"

In [ ]:
# 5. Learning curve: reward should rise from very negative towards 0
from IPython.display import Image, display
display(Image(filename="outputs/sac_pusher_1m/learning_curve.png"))

In [ ]:
# 6. Watch early, middle and final checkpoints on a TRAINING seed (OSMesa: EGL crashes once PyTorch is loaded)
!apt-get -qq install -y libosmesa6 > /dev/null
!python -m policies.watch_checkpoints --run outputs/sac_pusher_1m --ckpts ckpts/sac_20000_steps ckpts/sac_300000_steps ckpts/sac_600000_steps sac_final
import glob
from IPython.display import Image, display
for g in sorted(glob.glob("outputs/sac_pusher_1m/**/*.gif", recursive=True)):
    print(g); display(Image(filename=g))

In [ ]:
# 7. Measure pushing skill of all 50 checkpoints on 20 TRAINING seeds (a few minutes, no rendering)
!python -m policies.measure_checkpoints --run outputs/sac_pusher_1m
display(Image(filename="outputs/sac_pusher_1m/checkpoint_measure.png"))

In [ ]:
# 8. Zip to /kaggle/working (outside the repo folder, so re-cloning can't delete it). Download from the Output tab.
!cd outputs && zip -qr /kaggle/working/sac_pusher_1m.zip sac_pusher_1m
!ls -lh /kaggle/working/sac_pusher_1m.zip